# Multi-GPU JAX Setup for PyAutoLens

This notebook provides a comprehensive guide to setting up and configuring multi-GPU processing using JAX with PyAutoLens. Multi-GPU support allows you to significantly accelerate lens modeling computations by distributing the workload across multiple GPUs.

## Overview

- **Framework**: JAX (for GPU acceleration)
- **Application**: PyAutoLens (gravitational lens modeling)
- **Approach**: Multi-GPU parallelization using `pmap` and distributed arrays
- **Benefits**: Faster computations, parallel lens model fitting, reduced wall-clock time

## Prerequisites

Before starting, ensure you have:
- NVIDIA GPUs with CUDA support
- CUDA Toolkit (11.2 or higher recommended)
- cuDNN library installed
- Python 3.8+

## Step 1: Installation

First, install the required packages. Run the following commands in your terminal:

In [ ]:
# Installation commands (run in terminal)
"""
# For JAX with GPU support (CUDA):
pip install --upgrade pip
pip install jax[cuda11_cudnn82]=0.4.8

# For PyAutoLens:
pip install pyautolens

# For additional utilities:
pip install numpy scipy matplotlib

# Note: Ensure CUDA and cuDNN are installed system-wide before installing JAX
# For macOS with Metal acceleration (alternative):
# pip install jax[metal]
"""
print("Installation commands provided above - run these in your terminal")

## Section 1: Import Required Libraries

Import all necessary libraries for multi-GPU processing with JAX and PyAutoLens.

In [ ]:
import jax
import jax.numpy as jnp
from jax import pmap, vmap, jit
import numpy as np
import os
from typing import Tuple, Callable
import time

# Optional: Import PyAutoLens if available
try:
    import autolens as al
    import autofit as af
    AUTOLENS_AVAILABLE = True
except ImportError:
    AUTOLENS_AVAILABLE = False
    print("PyAutoLens not installed - core JAX multi-GPU features will still work")

print(f"JAX version: {jax.__version__}")
print(f"PyAutoLens available: {AUTOLENS_AVAILABLE}")

## Section 2: Check Available GPUs

Use JAX utilities to detect and display the number of available GPUs and their specifications.

In [ ]:
# Check available devices
devices = jax.devices()
print(f"Total devices: {len(devices)}")
print(f"Device type: {devices[0].platform}")
print(f"Devices: {devices}\n")

# Get device count per type
gpu_count = len(jax.devices('gpu'))
cpu_count = len(jax.devices('cpu'))

print(f"Number of GPUs: {gpu_count}")
print(f"Number of CPUs: {cpu_count}\n")

# Get device properties
print("Device Properties:")
for i, device in enumerate(jax.devices('gpu')):
    print(f"GPU {i}: {device}")
    
# Check JAX configuration
print("\nJAX Configuration:")
print(f"Platform: {jax.default_backend()}")

## Section 3: Configure JAX for Multi-GPU

Set up JAX environment variables and configuration for optimal multi-GPU performance.

### Key JAX Configuration Environment Variables

- `JAX_PLATFORM_NAME`: Set default platform (cpu, gpu, tpu)
- `JAX_DEVICES`: Specify which devices to use
- `CUDA_VISIBLE_DEVICES`: Control GPU visibility (set at system level)
- `JAX_DEBUG_PRINT_CACHE_MISSES`: Enable debug printing for cache analysis

In [ ]:
# Configure JAX for multi-GPU execution
# These should typically be set before JAX initialization

# Option 1: Set environment variables (already imported JAX, so these won't take effect)
# In production, set these BEFORE importing jax:
# os.environ['JAX_PLATFORM_NAME'] = 'gpu'
# os.environ['JAX_PLATFORMS'] = 'gpu'

# Option 2: Configure through jax.config
import jax.config as config

# Enable 64-bit precision for better numerical accuracy in lens modeling
config.update("jax_enable_x64", True)

# For debugging distributed operations (optional)
# config.update("jax_debug_print_cache_misses", True)

print("JAX Configuration Updated:")
print(f"  x64 precision enabled: {config.jax_enable_x64}")
print(f"  Platform: {jax.default_backend()}\n")

# Function to display optimal JAX settings for multi-GPU
def print_jax_settings():
    """Print current JAX settings optimized for multi-GPU."""
    print("=== JAX Multi-GPU Settings ===")
    print(f"Enable x64: {config.jax_enable_x64}")
    print(f"Available devices: {len(jax.devices())}")
    print(f"GPU devices: {len(jax.devices('gpu'))}")
    print(f"CPU devices: {len(jax.devices('cpu'))}")
    print(f"Default backend: {jax.default_backend()}")
    return config

print_jax_settings()

## Section 4: Multi-GPU Parallelization Techniques

JAX provides several approaches for multi-GPU parallelization. The most effective for lens modeling is `pmap` (parallel map).

In [ ]:
### Approach 1: Using pmap for Parallel Model Evaluation

# Define a simple function representing lens model computation
def compute_lens_model(params: jnp.ndarray) -> jnp.ndarray:
    """
    Simplified lens model computation.
    In practice, this would perform gravitational lensing calculations.
    """
    # Simulate lens model: some complex computation
    result = jnp.sin(params) + jnp.cos(params ** 2) + params ** 3
    return result

# Create a parallel version using pmap
# This maps the function across available GPUs
num_gpus = len(jax.devices('gpu')) if len(jax.devices('gpu')) > 0 else 1
print(f"Using {num_gpus} GPU(s) for parallelization")

# pmap version: automatically distributes across devices
pmap_compute_model = pmap(compute_lens_model)

# Test with data shaped for multiple GPUs
batch_size = num_gpus
test_params = jnp.ones((batch_size, 5))  # 5 parameters per model

print(f"Input shape: {test_params.shape}")
print(f"Expected output shape: ({batch_size}, 5)")

# Compute in parallel
parallel_result = pmap_compute_model(test_params)
print(f"Output shape: {parallel_result.shape}")
print(f"Parallel computation successful!\n")

In [ ]:
### Approach 2: Using vmap + jit for Batch Vectorization

# vmap automatically vectorizes functions across batch dimensions
def batched_lens_computation(params_batch):
    """Vectorized lens model computation using vmap."""
    # vmap will automatically map this function across the batch dimension
    return compute_lens_model(params_batch)

# Vectorize the function
vmapped_compute = vmap(compute_lens_model)
jitted_vmapped_compute = jit(vmapped_compute)  # Add JIT compilation

# Create a batch of parameters
num_models = 8  # Process 8 different models
batch_params = jnp.ones((num_models, 5))

# Compute using vectorized + JIT version
vmap_result = jitted_vmapped_compute(batch_params)
print(f"Vectorized computation:")
print(f"  Input shape: {batch_params.shape}")
print(f"  Output shape: {vmap_result.shape}")
print(f"  Vectorized + JIT computation successful!\n")

## Section 5: Initialize PyAutoLens with JAX Backend (if available)

Configure PyAutoLens to use JAX for GPU acceleration.

In [ ]:
if AUTOLENS_AVAILABLE:
    print("=== PyAutoLens JAX Configuration ===\n")
    
    # PyAutoLens typically uses NumPy by default
    # To leverage GPU acceleration, wrap computations with JAX functions
    
    print("PyAutoLens loaded successfully.")
    print("Note: PyAutoLens uses NumPy arrays internally.")
    print("To accelerate, convert to JAX arrays and use vmap/pmap wrappers.\n")
    
    # Example: Create a simple galaxy model
    # (Requires full PyAutoLens installation)
    try:
        # Create a simple lens galaxy
        lens_galaxy = al.Galaxy(
            redshift=0.5,
            mass=al.mp.Isothermal(centre=(0.0, 0.0), einstein_radius=1.0)
        )
        print(f"Successfully created lens galaxy: {lens_galaxy}")
    except Exception as e:
        print(f"Could not create galaxy model: {e}")
        print("(This is normal if PyAutoLens is not fully installed)")
else:
    print("PyAutoLens not available in this environment.")
    print("However, JAX GPU acceleration is ready for custom lens modeling code.\n")

## Section 6: Advanced Multi-GPU Pattern - Distributed Batch Processing

Implement a practical example of distributed lens model evaluation across multiple GPUs.

In [ ]:
class MultiGPULensModelEvaluator:
    """
    Multi-GPU parallelized lens model evaluator using JAX.
    This class demonstrates best practices for GPU-accelerated lens modeling.
    """
    
    def __init__(self, num_gpus: int = None):
        """
        Initialize the evaluator.
        
        Args:
            num_gpus: Number of GPUs to use. If None, use all available.
        """
        self.available_gpus = len(jax.devices('gpu')) if len(jax.devices('gpu')) > 0 else 1
        self.num_gpus = num_gpus or self.available_gpus
        print(f"MultiGPU Evaluator initialized with {self.num_gpus} GPU(s)")
        
    def lens_model_function(self, params: jnp.ndarray) -> jnp.ndarray:
        """
        Define a lens model computation.
        
        In real applications, this would implement:
        - Lens potential evaluation
        - Light profile computation
        - Lensing equation solving
        - PSF convolution
        """
        # Simplified model: simulate lens physics computation
        # params shape: (n_params,)
        alpha = params[0]  # Convergence parameter
        beta = params[1]   # Shear parameter
        
        # Compute deflection angle (simplified)
        deflection = alpha * jnp.sqrt(params[2:].sum() + 1e-10)
        
        # Compute effective potential
        potential = (alpha ** 2 + beta ** 2) * deflection
        
        return potential
    
    def evaluate_batch_parallel(self, param_batch: jnp.ndarray) -> jnp.ndarray:
        """
        Evaluate multiple lens models in parallel across GPUs.
        
        Args:
            param_batch: Array of shape (n_models, n_params) or (n_gpus, n_models_per_gpu, n_params)
            
        Returns:
            Array of lens model evaluations
        """
        # Reshape for GPU distribution if needed
        if param_batch.ndim == 2:
            n_models = param_batch.shape[0]
            n_params = param_batch.shape[1]
            
            # Pad to multiple of num_gpus
            pad_size = (self.num_gpus - n_models % self.num_gpus) % self.num_gpus
            padded_params = jnp.pad(
                param_batch, 
                ((0, pad_size), (0, 0)),
                mode='edge'
            )
            
            # Reshape for pmap distribution
            reshaped = padded_params.reshape(-1, self.num_gpus, n_params)
            
            # Apply pmap across first dimension (GPU distribution)
            pmapped_fn = pmap(jit(vmap(self.lens_model_function)))
            results = pmapped_fn(reshaped)
            
            # Reshape back and remove padding
            return results.reshape(-1)[:n_models]
        else:
            # Already shaped correctly
            pmapped_fn = pmap(jit(vmap(self.lens_model_function)))
            return pmapped_fn(param_batch)
    
    def benchmark_performance(self, n_models: int = 1000, n_params: int = 10):
        """
        Benchmark the multi-GPU performance.
        
        Args:
            n_models: Number of models to evaluate
            n_params: Number of parameters per model
        """
        print(f"\n=== Performance Benchmark ===")
        print(f"Models to evaluate: {n_models}")
        print(f"Parameters per model: {n_params}")
        print(f"GPUs used: {self.num_gpus}\n")
        
        # Generate random parameters
        key = jax.random.PRNGKey(0)
        params = jax.random.normal(key, (n_models, n_params))
        
        # Warm-up
        _ = self.evaluate_batch_parallel(params[:10])
        
        # Benchmark parallel execution
        start = time.time()
        results_parallel = self.evaluate_batch_parallel(params)
        parallel_time = time.time() - start
        
        # Single GPU execution for comparison
        single_gpu_fn = jit(vmap(self.lens_model_function))
        start = time.time()
        results_single = single_gpu_fn(params)
        single_time = time.time() - start
        
        # Print results
        speedup = single_time / parallel_time if parallel_time > 0 else float('inf')
        
        print(f"Single GPU time: {single_time:.4f} seconds")
        print(f"Multi-GPU time: {parallel_time:.4f} seconds")
        print(f"Speedup: {speedup:.2f}x")
        print(f"Throughput: {n_models / parallel_time:.0f} models/sec")
        
        return {
            'single_gpu_time': single_time,
            'multi_gpu_time': parallel_time,
            'speedup': speedup,
            'throughput': n_models / parallel_time
        }

# Create evaluator instance
evaluator = MultiGPULensModelEvaluator()

# Run benchmark
benchmark_results = evaluator.benchmark_performance(n_models=1000, n_params=10)
print("\nBenchmark completed successfully!")

## Section 7: Optimization Tips and Best Practices

### Memory Management
- **Data sharding**: Split data across GPUs to manage memory constraints
- **Asynchronous execution**: Use `jit` with `device_put` for better memory utilization
- **Gradient accumulation**: For large parameter spaces, accumulate gradients across batches

In [ ]:
print("=== Multi-GPU Best Practices ===\n")

best_practices = """
1. DEVICE PLACEMENT
   - Use jax.device_put() to explicitly place arrays on specific devices
   - Monitor with jax.debug.print_ir() to verify computation graph

2. PARALLELIZATION STRATEGY
   - Use pmap for coarse-grained parallelism (across GPUs)
   - Use vmap for fine-grained parallelism (across data dimensions)
   - Combine both for maximum efficiency

3. DATA SHARDING
   - Divide input data by number of GPUs
   - Use replicated vs sharded arrays appropriately
   - Monitor memory usage with nvidia-smi

4. GRADIENT COMPUTATION
   - Use jax.grad() inside vmap for efficient batched gradients
   - Consider numerical precision (x64) for lens modeling

5. SYNCHRONIZATION
   - Block on device operations when timing is critical
   - Use jax.numpy.block_until_ready() for synchronization

6. PROFILING & DEBUGGING
   - Enable JAX debug logging: os.environ['JAX_DEBUG_PRINT_CACHE_MISSES'] = '1'
   - Use jax.profiler for detailed performance analysis
   - Monitor GPU memory with nvidia-smi or similar tools
"""

print(best_practices)

# Example: Device placement
print("\n=== Example: Explicit Device Placement ===\n")

def place_on_devices(arrays, devices=None):
    """Place arrays on specific devices."""
    if devices is None:
        devices = jax.devices('gpu')
    
    placed_arrays = []
    for i, arr in enumerate(arrays):
        device_idx = i % len(devices)
        placed_arr = jax.device_put(arr, devices[device_idx])
        placed_arrays.append(placed_arr)
    return placed_arrays

# Create sample data
sample_data = [jnp.ones((100, 100)) for _ in range(4)]

# Place on devices
if len(jax.devices('gpu')) > 0:
    device_placed = place_on_devices(sample_data)
    print(f"Placed {len(device_placed)} arrays on {len(jax.devices('gpu'))} GPU(s)")
else:
    print("No GPUs available for placement demonstration")


## Section 8: Troubleshooting and Common Issues

### Common Issues and Solutions

In [ ]:
troubleshooting_guide = """
┌─────────────────────────────────────────────────────────────────────────┐
│                    TROUBLESHOOTING GUIDE                                 │
└─────────────────────────────────────────────────────────────────────────┘

Issue 1: "No GPU devices found"
─────────────────────────────────
Problem: JAX cannot detect CUDA devices
Solutions:
  • Check CUDA installation: nvidia-smi
  • Verify JAX CUDA version matches your CUDA: pip show jax
  • Install correct JAX GPU package: pip install jax[cuda11_cudnn82]
  • Set CUDA paths: export CUDA_PATH=/usr/local/cuda
  • Reinstall JAX: pip install --upgrade --force-reinstall jax[cuda11_cudnn82]

Issue 2: "pmap requires equal-sized arrays"
─────────────────────────────────────────────
Problem: Input array size not divisible by number of devices
Solutions:
  • Pad input arrays: jnp.pad(array, ((0, n_gpus - n % n_gpus), (0, 0)))
  • Reshape data to (n_gpus, batch_per_gpu, ...)
  • Use dynamic batching with conditional logic
  • Implement data loading that respects GPU count

Issue 3: "Out of memory" errors
────────────────────────────────
Problem: GPU memory exhausted during computation
Solutions:
  • Reduce batch size per GPU
  • Use 32-bit precision instead of 64-bit: config.update("jax_enable_x64", False)
  • Enable memory growth: os.environ['XLA_FLAGS'] = '--xla_gpu_enable_persistent_instrs_cache=false'
  • Profile memory usage: jax.profiler.start_trace('trace.gz')
  • Split computation into smaller chunks
  • Use gradient checkpointing for backward passes

Issue 4: "Slow multi-GPU performance"
──────────────────────────────────────
Problem: Multi-GPU not faster than single GPU
Solutions:
  • Verify computation actually runs on GPUs: jax.default_backend()
  • Check GPU utilization: nvidia-smi dmon
  • Reduce host↔device data transfer
  • Use jit compilation: @jit
  • Ensure batch size is large enough (typically ≥ 128 per GPU)
  • Profile with: jax.profiler.trace(lambda: fn(args), _r=150)
  • Consider memory bandwidth limitations

Issue 5: "Numerical precision issues"
──────────────────────────────────────
Problem: Results differ between single/multi-GPU or CPU/GPU
Solutions:
  • Enable 64-bit precision: config.update("jax_enable_x64", True)
  • Check numerical stability in lens model equations
  • Compare results: jnp.allclose(result1, result2, rtol=1e-5)
  • Use stable mathematical operations (avoid subtraction of large numbers)

Issue 6: "Hanging or deadlock"
──────────────────────────────
Problem: Program hangs or appears to freeze
Solutions:
  • Check for data dependency issues in pmap
  • Ensure pmap input has correct shape: (n_devices, ...)
  • Add timeout in long-running computations
  • Use jax.experimental.mesh_utils.create_device_mesh for advanced cases
  • Check for GPU compute capability issues

Issue 7: "pmap and GPU topology"
────────────────────────────────
Problem: pmap not distributing across all GPUs
Solutions:
  • Verify device count: len(jax.devices('gpu'))
  • Check physical topology: nvidia-smi topo -m
  • Set device visibility: CUDA_VISIBLE_DEVICES=0,1,2,3
  • Use jax.experimental.mesh for advanced topologies
  • Monitor with: nvidia-smi dmon -s pucvmet

┌─────────────────────────────────────────────────────────────────────────┐
│                    VERIFICATION CHECKLIST                               │
└─────────────────────────────────────────────────────────────────────────┘

□ CUDA Toolkit installed and in PATH
□ cuDNN library installed
□ JAX version matches CUDA version
□ Multiple GPUs detected by CUDA
□ JAX detects all GPUs: len(jax.devices('gpu')) > 1
□ Input data properly shaped for pmap
□ Batch size optimized for GPU memory
□ x64 precision enabled for lens modeling
□ GPU memory monitoring setup
□ JIT compilation working correctly
□ Gradients computed successfully (if using optimization)
"""

print(troubleshooting_guide)

## Section 9: Verification and Final Checklist

Run the following checks to verify your multi-GPU setup is working correctly.

In [ ]:
def run_verification_suite():
    """Run comprehensive verification of multi-GPU setup."""
    
    print("\n" + "="*60)
    print("MULTI-GPU JAX VERIFICATION SUITE")
    print("="*60 + "\n")
    
    results = {}
    
    # Check 1: Device detection
    print("✓ Check 1: Device Detection")
    try:
        devices = jax.devices()
        gpu_devices = jax.devices('gpu')
        cpu_devices = jax.devices('cpu')
        print(f"  Total devices: {len(devices)}")
        print(f"  GPU devices: {len(gpu_devices)}")
        print(f"  CPU devices: {len(cpu_devices)}")
        results['device_detection'] = True
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        results['device_detection'] = False
    
    # Check 2: JAX configuration
    print("\n✓ Check 2: JAX Configuration")
    try:
        print(f"  Backend: {jax.default_backend()}")
        print(f"  x64 enabled: {config.jax_enable_x64}")
        results['jax_config'] = True
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        results['jax_config'] = False
    
    # Check 3: Basic computation
    print("\n✓ Check 3: Basic JAX Computation")
    try:
        test_array = jnp.ones((1000, 1000))
        result = jnp.dot(test_array, test_array)
        print(f"  Matrix multiplication successful")
        print(f"  Result shape: {result.shape}")
        results['basic_compute'] = True
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        results['basic_compute'] = False
    
    # Check 4: JIT compilation
    print("\n✓ Check 4: JIT Compilation")
    try:
        @jit
        def jitted_func(x):
            return jnp.sin(x) + jnp.cos(x)
        
        test_input = jnp.linspace(0, 2*jnp.pi, 100)
        output = jitted_func(test_input)
        print(f"  JIT compilation successful")
        print(f"  Output shape: {output.shape}")
        results['jit_compile'] = True
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        results['jit_compile'] = False
    
    # Check 5: vmap functionality
    print("\n✓ Check 5: Vectorization (vmap)")
    try:
        def single_func(x):
            return x ** 2 + jnp.sin(x)
        
        vmapped_func = vmap(single_func)
        batch_input = jnp.ones((10, 100))
        output = vmapped_func(batch_input)
        print(f"  vmap successful")
        print(f"  Input shape: {batch_input.shape} → Output shape: {output.shape}")
        results['vmap'] = True
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        results['vmap'] = False
    
    # Check 6: pmap functionality (if multi-GPU)
    print("\n✓ Check 6: Parallelization (pmap)")
    try:
        if len(jax.devices('gpu')) > 0:
            def pmap_func(x):
                return jnp.sum(x)
            
            pmapped_func = pmap(pmap_func)
            num_gpus = len(jax.devices('gpu'))
            pmap_input = jnp.ones((num_gpus, 100, 100))
            output = pmapped_func(pmap_input)
            print(f"  pmap successful on {num_gpus} GPU(s)")
            print(f"  Input shape: {pmap_input.shape} → Output shape: {output.shape}")
            results['pmap'] = True
        else:
            print(f"  Skipped: No GPUs available")
            results['pmap'] = None
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        results['pmap'] = False
    
    # Check 7: Gradient computation
    print("\n✓ Check 7: Automatic Differentiation")
    try:
        def loss_func(x):
            return jnp.sum(x ** 2)
        
        grad_func = jax.grad(loss_func)
        test_input = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
        grads = grad_func(test_input)
        print(f"  Gradient computation successful")
        print(f"  Gradients shape: {grads.shape}")
        results['autodiff'] = True
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        results['autodiff'] = False
    
    # Summary
    print("\n" + "="*60)
    print("VERIFICATION SUMMARY")
    print("="*60)
    
    passed = sum(1 for v in results.values() if v is True)
    total = len([v for v in results.values() if v is not None])
    
    for check, status in results.items():
        status_str = "✓ PASS" if status is True else ("✗ FAIL" if status is False else "⊘ SKIP")
        print(f"  {status_str}: {check}")
    
    print(f"\nResult: {passed}/{total} checks passed")
    
    if passed == total:
        print("\n✓ Multi-GPU JAX setup is ready for use!")
    else:
        print("\n⚠ Some checks failed. See above for details.")
    
    return results

# Run verification
verification_results = run_verification_suite()

## Section 10: Next Steps and Additional Resources

### For PyAutoLens Integration

1. **Wrap PyAutoLens computations with JAX**
   - Convert lens models to JAX-compatible functions
   - Use vmap/pmap for batched model evaluation
   - Integrate into search/inference pipelines

2. **Performance optimization**
   - Profile your specific lens models
   - Identify bottlenecks with jax.profiler
   - Tune batch sizes and memory usage

3. **Advanced techniques**
   - Gradient-based optimization with jax.grad
   - Custom CUDA kernels with jaxlib.xla_extension
   - Distributed training across multiple nodes

### Useful Links and Resources

- **JAX Documentation**: https://jax.readthedocs.io/
- **JAX GPU Setup**: https://jax.readthedocs.io/en/latest/installation.html
- **PyAutoLens Documentation**: https://pyautolens.readthedocs.io/
- **JAX Parallelization Guide**: https://jax.readthedocs.io/en/latest/parallel-evaluation.html
- **NVIDIA GPU Computing**: https://developer.nvidia.com/cuda-toolkit

### Common PyAutoLens + JAX Patterns

See the `MultiGPULensModelEvaluator` class above for a template. Key patterns:

- **Batch model evaluation**: Use vmap to evaluate multiple models simultaneously
- **Parallel grid search**: Use pmap to distribute across GPUs
- **Gradient-based fitting**: Use jax.grad within vmapped/pmapped functions
- **Result aggregation**: Combine results across devices for analysis

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════╗
║                    SETUP COMPLETE!                                     ║
╚════════════════════════════════════════════════════════════════════════╝

Your multi-GPU JAX setup for PyAutoLens is now configured.

QUICK START GUIDE:
─────────────────

1. Install packages (if not already done):
   pip install jax[cuda11_cudnn82] pyautolens numpy scipy matplotlib

2. Run verification (above) to ensure setup is working

3. Use the MultiGPULensModelEvaluator class for your lens modeling tasks

4. Scale up with your own data and models

PERFORMANCE TIPS:
────────────────

• Start with smaller batch sizes and increase incrementally
• Monitor GPU utilization: nvidia-smi -l 1
• Use jax.profiler for bottleneck identification
• Enable 64-bit precision for accurate lens modeling
• Combine vmap + pmap for hierarchical parallelism

GETTING HELP:
─────────────

• JAX issues: https://github.com/google/jax/issues
• PyAutoLens issues: https://github.com/Jammy2211/PyAutoLens/issues
• CUDA issues: Check NVIDIA documentation and nvidia-smi output

═══════════════════════════════════════════════════════════════════════════
""")

print("Notebook setup complete! You're ready to use multi-GPU JAX with PyAutoLens.")